# Camera-depth training and validation analysis

Ce notebook lance optionnellement un entraînement KITTI, charge les métriques par époque, compare le modèle à la profondeur splattée et inspecte une prédiction du split de validation. La logique réutilisable reste dans `rgb_lidar_fusion.camera_depth_analysis`.

In [ ]:
from pathlib import Path
import sys
from IPython.display import display

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from rgb_lidar_fusion.camera_depth_analysis import (
    analyze_camera_depth_sample,
    has_camera_depth_validation_metrics,
    load_camera_depth_checkpoint,
    load_camera_depth_metrics,
    plot_camera_depth_metrics,
    plot_camera_depth_sample,
    require_gpu_ready,
    run_gpu_preflight,
)
from rgb_lidar_fusion.dataset import KittiObjectDepthDataset
from rgb_lidar_fusion.lidar_splatting import SplattingConfig
from rgb_lidar_fusion.training import (
    load_training_config,
    run_kitti_camera_depth_training,
    split_dataset_indices,
)

## Paramètres du run

L'entraînement est désactivé par défaut. Passer `RUN_TRAINING` à `True` pour créer le run d'analyse. `EPOCHS` est le nombre total d'époques visé, y compris lors d'une reprise. Le batch notebook reste à 1 par défaut pour conserver une marge sur le GPU local de 4 Gio. Avec `REQUIRE_CUDA=True`, un préflight valide est obligatoire avant l'entraînement.

In [ ]:
CONFIG_PATH = ROOT / "configs/training/kitti_camera_depth.yaml"
RUN_OUTPUT_DIR = ROOT / "results/training/kitti_camera_depth_notebook"
RUN_TRAINING = False
RESUME_EXISTING = False
EPOCHS = 5
BATCH_SIZE = 1
DEVICE = "auto"
REQUIRE_CUDA = True
MINIMUM_FREE_GPU_MEMORY_MIB = 2500
VALIDATION_SAMPLE_POSITION = 0

config = load_training_config(CONFIG_PATH)
data_root = Path(str(config["data_root"]))
if not data_root.is_absolute():
    config["data_root"] = str((ROOT / data_root).resolve())
config.update(
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    device=DEVICE,
    output_dir=str(RUN_OUTPUT_DIR),
)
checkpoint_path = RUN_OUTPUT_DIR / "checkpoints/latest.pt"
metrics_path = RUN_OUTPUT_DIR / "metrics.json"
config

## Préflight GPU

Ce contrôle vérifie l'état de récupération NVIDIA, la mémoire libre, les autres processus CUDA et exécute un petit calcul CUDA synchronisé. Si le rapport échoue, ne pas lancer l'entraînement ; une action `Reset` après une erreur Xid nécessite de redémarrer la machine.

In [ ]:
gpu_report = None
if DEVICE != "cpu":
    gpu_report = run_gpu_preflight(
        minimum_free_memory_mib=MINIMUM_FREE_GPU_MEMORY_MIB,
        run_smoke_test=True,
    )
    print(gpu_report.format())
else:
    print("GPU preflight skipped because DEVICE='cpu'.")

In [ ]:
if RUN_TRAINING and REQUIRE_CUDA and DEVICE != "cpu":
    if gpu_report is None:
        raise RuntimeError("Run the GPU preflight cell before training.")
    require_gpu_ready(gpu_report)

if RUN_TRAINING:
    run_config = dict(config)
    if RESUME_EXISTING:
        if not checkpoint_path.is_file():
            raise FileNotFoundError(f"Cannot resume missing checkpoint: {checkpoint_path}")
        run_config["resume_from"] = str(checkpoint_path)
    result = run_kitti_camera_depth_training(run_config)
    print(result.device_diagnostic)
    print(f"epochs_completed={result.epochs_completed}")
    print(f"checkpoint={result.checkpoint_path}")
else:
    print("Training skipped. Set RUN_TRAINING=True to create or extend the analysis run.")

## Courbes et comparaison au splatting

La loss est exprimée en profondeur normalisée. Les MAE/RMSE sont exprimées en mètres sur les mêmes pixels LiDAR holdout pour le réseau et la baseline `depth_expanded`.

In [ ]:
metrics = []
if metrics_path.is_file():
    metrics = load_camera_depth_metrics(metrics_path)
    if has_camera_depth_validation_metrics(metrics):
        metric_figure = plot_camera_depth_metrics(metrics)
        display(metric_figure)
    else:
        print(
            "This run predates validation metrics. Create a new run with "
            "RUN_TRAINING=True before plotting train/validation curves."
        )
else:
    print(f"No metrics yet at {metrics_path}")

In [ ]:
if has_camera_depth_validation_metrics(metrics):
    latest = metrics[-1]
    mae_gain_m = latest["val_splat_mae_m"] - latest["val_mae_m"]
    rmse_gain_m = latest["val_splat_rmse_m"] - latest["val_rmse_m"]
    print(f"epoch={latest['epoch']}")
    print(f"model MAE={latest['val_mae_m']:.3f} m | splat MAE={latest['val_splat_mae_m']:.3f} m")
    print(f"model RMSE={latest['val_rmse_m']:.3f} m | splat RMSE={latest['val_splat_rmse_m']:.3f} m")
    print(f"gain over splat: MAE={mae_gain_m:+.3f} m, RMSE={rmse_gain_m:+.3f} m")
    print(f"splat coverage={latest['val_splat_coverage']:.1%} | validation pixels={latest['val_pixel_count']}")

## Inspection d'une frame de validation

Cette vue reconstruit exactement le holdout déterministe utilisé en validation. L'erreur affichée n'existe qu'aux pixels LiDAR retirés des entrées.

In [ ]:
if checkpoint_path.is_file():
    loaded = load_camera_depth_checkpoint(checkpoint_path, device=DEVICE)
    dataset = KittiObjectDepthDataset(
        ROOT / str(config["data_root"]),
        image_shape=(int(config["height"]), int(config["width"])),
        max_depth_m=float(config["max_depth_m"]),
        sample_limit=int(config["sample_count"]),
    )
    _, validation_indices = split_dataset_indices(
        len(dataset),
        validation_fraction=float(config["validation_fraction"]),
        seed=int(config["split_seed"]),
    )
    dataset_index = validation_indices[VALIDATION_SAMPLE_POSITION]
    analysis = analyze_camera_depth_sample(
        model=loaded.model,
        dataset=dataset,
        dataset_index=dataset_index,
        holdout_fraction=float(config["holdout_fraction"]),
        holdout_seed=int(config["validation_holdout_seed"]),
        splatting_config=SplattingConfig(
            radius_px=int(config["splat_radius_px"]),
            sigma_px=float(config["splat_sigma_px"]),
        ),
        device=loaded.device,
        max_depth_m=float(config["max_depth_m"]),
    )
    print(f"checkpoint epoch={loaded.epoch}, step={loaded.step}, sample={analysis.sample_id}")
    sample_figure = plot_camera_depth_sample(analysis)
    display(sample_figure)
else:
    print(f"No checkpoint yet at {checkpoint_path}")

## Lecture des résultats

- `loss` et `val_loss` qui diminuent ensemble indiquent un apprentissage cohérent sur les points supervisés.
- Une loss train qui baisse tandis que la validation stagne ou remonte indique un surapprentissage.
- Un gain MAE/RMSE positif signifie que le réseau bat la profondeur splattée sur les pixels holdout.
- Ces mesures restent sparse : elles ne prouvent pas la qualité de la profondeur aux pixels dépourvus de vérité terrain LiDAR.